In [ ]:
import torch
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
transform = transforms.Compose([transforms.ToTensor()])
trainset_tmp = datasets.MNIST('~/', download=True, train=True, transform=transform)
trainloader_tmp = torch.utils.data.DataLoader(trainset_tmp, batch_size=10_000, shuffle=True)
testset = datasets.MNIST('~/', download=True, train=False, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=512, shuffle=False)


In [ ]:
import torch
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
transform = transforms.Compose([transforms.ToTensor()])
trainset_tmp = datasets.MNIST('~/', download=True, train=True, transform=transform)
trainloader_tmp = torch.utils.data.DataLoader(trainset_tmp, batch_size=10_000, shuffle=True)
testset = datasets.MNIST('~/', download=True, train=False, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=512, shuffle=False)

In [ ]:
def data_divides__minst():
    Train = []
    for image in trainset_tmp:
        Ima = image[0][0].reshape(1,784)
        if image[1]%2 == 0:
            number = 1
        elif image[1]%2 == 1:
            number = -1
        Train.append([np.array(Ima[0][:]),number])
    return Train
Train = data_divides__minst()

def data_divides__minst_test():
    Test = []
    for image in testset:
        Ima = image[0][0].reshape(1,784)
        if image[1]%2 == 0:
            number = 1
        elif image[1]%2 == 1:
            number = -1
        Test.append([np.array(Ima[0][:]),number])
    return Test
Test = data_divides__minst_test()
#训练集测试集合处理为球面数据
Test_data = []
for image in Test:
    image_pic = image[0]/10
    a = 2/(sum(image_pic*image_pic)+1)
    Ima = np.append(image_pic*a, a-1)
    Test_data.append([Ima[:],image[1]])

In [ ]:
import numpy as np
import math
import random
import time
class Kernel_SGD():
    def __init__(self,class_num,d,s):
        self.class_num = class_num
        self.d = d
        self.Alpha_last = np.zeros((1,self.class_num))
        self.Alpha_ave = np.zeros((1,self.class_num))
        self.X = [np.zeros(self.d)]
        self.s1 = s

    def Matern_kernel(self,x1,x2):
        t = x1-x2
        tau = np.dot(t,t)/(2*400)
        return np.exp(-tau)

    def computation_f_last(self,x):
        S = np.zeros(self.class_num)
        if len(self.Alpha_last) > 0:
            A = np.array([ self.Matern_kernel(a,x) for a in self.X])
            S = np.dot(A,self.Alpha_last)
        return S

    def computation_f_ave(self,x):
        S = np.zeros(self.class_num)
        if len(self.Alpha_ave) > 0:
            A = np.array([ self.Matern_kernel(a,x) for a in self.X])
            S = np.dot(A,self.Alpha_ave)
        return S
            
    def updata(self,x,y,epoch):
        self.gamma = self.gamma0*(epoch**(-self.t))
        exp = np.exp(-y*self.computation_f_last(x))
        self.Alpha_last = np.append(self.Alpha_last,(self.gamma*exp/(exp+1)*y).reshape(1,self.class_num),axis =0)
        self.Alpha_ave = np.append(self.Alpha_ave, np.zeros((1,self.class_num)), axis = 0)
        self.Alpha_ave = (epoch*self.Alpha_ave +self.Alpha_last)/(epoch+1)
        self.X.append(x)

    def Data_collect(self):
        x, y = random.sample(Train, 1)[0]
        noise = np.random.normal(0, 0.0005, x.shape)
        return [x + noise, y]

    def test(self):
        S = 0
        Accuarcy = 0
        for i in range(1000):
            data = random.sample(Test,1)[0]
            y = self.computation_f_ave(data[0])
            #print(y)
            S += np.dot(y-data[1],y-data[1])
            if y*data[1] > 0:
                Accuarcy += 1
        return [S/1000,Accuarcy/1000]
    
    def algorthim(self,t,gamma0,Epoch,NN,operation):
        self.Accuracy = []
        self.Error = []
        self.Time = []
        start = time.perf_counter()
        self.t = t
        self.gamma0 = gamma0
        for epoch in range(Epoch):
            Z = self.Data_collect()
            self.updata(Z[0],Z[1],epoch)
            if epoch in NN:
                if operation == 'time':
                    end = time.perf_counter()
                    runTime = end - start
                    self.Time.append(runTime)
                    print(epoch,runTime)
                elif operation == 'test':
                    err,acc = self.test()
                    self.Accuracy.append(acc)
                    self.Error.append(err)
                    print(epoch,acc,err)

        return [self.Accuracy,self.Error]
            
        
        

In [ ]:
N = [int(10**(i/5))for i in range(1,36)]
class_num = 1
d = 784
Time = []
t = 0.0
gamma = 0.1
Acc = []
for i in range(1):
    kernel_sgd = Kernel_SGD(class_num,d,0.01+i)
    accracy,error = kernel_sgd.algorthim(0,0.1,100001,N,'test')
    Acc.append(kernel_sgd.Accuracy[:])

In [ ]:
N = [int(10**(i/5))for i in range(1,36)]
class_num = 1
d = 784
Time = []
t = 0.0
gamma = 0.1
Time = []
for i in range(1):
    kernel_sgd = Kernel_SGD(class_num,d,0.01+i)
    accracy,error = kernel_sgd.algorthim(0,0.1,100001,N,'time')
    Time.append(kernel_sgd.Time[:])